# Joint mass through chance children

Study exact mass propagation with no betting model and no posterior normalization.
Start with two private-hand pairs, follow the next card, and verify that every pair's mass is conserved across all its continuations.

The core operation is `propagate_joint_mass(state, joint_mass)`. It yields one street at a time and retains labelled children with zero mass. Start with small supports: retaining many large child masses can still be expensive.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'experiments' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from collections import defaultdict
from fractions import Fraction
import pandas as pd
from cards import card_name, make_hand, parse_card
from chance import ChanceState
from hand_space import HAND_INDEX
from mass import JointEntry, JointMass, propagate_joint_mass

## 1. Choose starting mass

Give 44 units to one deal and 88 to another. These are experimental weights.
Only the two listed pairs carry mass. The cross-pairs carry zero, even though they are legal: this input cannot be reconstructed as a product of its marginal weights.

Each compatible turn pair has 44 legal rivers, so these weights make the first experiment easy to inspect: it contributes one or two units to each legal river.

In [2]:
turn = make_hand(['2c', '7d', '9h', 'Js'])
pair_one = (make_hand(['Jc', 'Ad']), make_hand(['9c', 'Kd']))
pair_two = (make_hand(['4d', '5d']), make_hand(['8c', 'Qd']))
keys = [tuple(HAND_INDEX[hand] for hand in pair) for pair in (pair_one, pair_two)]
parent = JointMass((JointEntry(*keys[0], 44), JointEntry(*keys[1], 88)))

pd.DataFrame([
    {'pair': 'one', 'first': 'Jc Ad', 'second': '9c Kd', 'starting mass': 44},
    {'pair': 'two', 'first': '4d 5d', 'second': '8c Qd', 'starting mass': 88},
])

,pair,first,second,starting mass
0,one,Jc Ad,9c Kd,44
1,two,4d 5d,8c Qd,88


## 2. Distribute mass over all rivers

The public turn has 48 candidate river labels. Each pair contributes to 44 of them.
For example, Ad blocks pair one, 4d blocks pair two, and 6s blocks neither.
The private hands remain fixed; their mass is divided among labelled continuations.

In [3]:
children = tuple(propagate_joint_mass(ChanceState(turn), parent))
assert len(children) == 48

rows = []
for child in children:
    weights = {(entry.first_index, entry.second_index): entry.weight
               for entry in child.joint_mass.entries}
    rows.append({
        'river': card_name(child.reveal),
        'pair one mass': weights.get(keys[0], Fraction()),
        'pair two mass': weights.get(keys[1], Fraction()),
        'total mass': child.joint_mass.total,
    })
river_table = pd.DataFrame(rows)
river_table[river_table['river'].isin(['Ad', 'Kd', '4d', 'Qd', '6s'])]

,river,pair one mass,pair two mass,total mass
14,4d,1,0,1
21,Qd,1,0,1
22,Kd,0,2,2
23,Ad,0,2,2
40,6s,1,2,3


## 3. Recover the starting mass

Add contributions across all rivers separately for each private-hand pair.
Exact equality at every coordinate is stronger than checking only the total.
No child was normalized.

In [4]:
recovered = defaultdict(Fraction)
for child in children:
    for entry in child.joint_mass.entries:
        recovered[(entry.first_index, entry.second_index)] += entry.weight
expected = {(entry.first_index, entry.second_index): entry.weight
            for entry in parent.entries}
assert dict(recovered) == expected
assert sum((child.joint_mass.total for child in children), Fraction()) == parent.total

pd.DataFrame([
    {'pair': label, 'starting mass': expected[key], 'recovered mass': recovered[key]}
    for label, key in zip(['one', 'two'], keys)
])

,pair,starting mass,recovered mass
0,one,44,44
1,two,88,88


## 4. Continue from a flop through one selected turn

Now place the same starting mass on the flop 2c 7d 9h and select the Js turn.
Each deal has 45 legal turns, so this branch receives 1/45 of the starting mass.
Feed its state and its unnormalised mass into the same operation to enumerate rivers.

All rivers under that selected turn sum to the selected turn's mass. The other turn branches retain the remainder; following one branch does not reset its mass to one.

In [5]:
flop = make_hand(['2c', '7d', '9h'])
flop_state = ChanceState(flop, reveals=(flop,))
selected_turn = next(
    child for child in propagate_joint_mass(flop_state, parent)
    if child.reveal == parse_card('Js')
)
next_children = tuple(propagate_joint_mass(selected_turn.state, selected_turn.joint_mass))
selected_river = next(child for child in next_children if child.reveal == parse_card('6s'))
assert selected_turn.joint_mass.total == parent.total / 45
assert sum((child.joint_mass.total for child in next_children), Fraction()) == selected_turn.joint_mass.total
assert selected_river.joint_mass.total == parent.total / (45 * 44)
assert selected_river.state.reveals == (flop, parse_card('Js'), parse_card('6s'))

pd.DataFrame([
    {'stage': 'all starting flop mass', 'mass': parent.total},
    {'stage': 'selected Js turn', 'mass': selected_turn.joint_mass.total},
    {'stage': 'all rivers under Js', 'mass': sum((c.joint_mass.total for c in next_children), Fraction())},
    {'stage': 'selected Js then 6s path', 'mass': selected_river.joint_mass.total},
])

,stage,mass
0,all starting flop mass,132
1,selected Js turn,44/15
2,all rivers under Js,44/15
3,selected Js then 6s path,1/15


## What to vary next

Change the two starting weights, add a compatible pair, or select a different reveal. Conservation still holds, but the mass assigned to each child can change.

An empty `JointMass(())` propagates to empty children. A river has no further chance children. Positive-mass deals that overlap the board, overlap each other, or contradict fixed hands in `ChanceState` are rejected when the iterator is consumed.

To calculate a branch probability later, divide its total mass by the positive parent total. To interpret a nonempty child's mass as a conditional belief, call `.normalized()` explicitly. Neither operation is required for these experiments.